---
## 6b. Coordinate check — using CSV values directly

Your CSV coordinates are already in the **original frame resolution**, so no
scaling is needed.

The cell below confirms this by checking that labels span the full frame and
that the §8 overlay will show the disc sitting on the ball. It sets `SCALE = 1.0`
and leaves every label unchanged.

If for any reason the overlay in §8 shows the disc in the wrong place, come back
here and change `SCALE` to `frame_width / annotation_width`
(e.g. `1920 / 512 = 3.75`).


# ---- confirm coordinates match the frame size ----------------------------
from PIL import Image as _PIL

_df_check = pd.read_csv(CSV_PATH)
_vx = _df_check.loc[_df_check["Visibility"] == 1, "X"].tolist()
_vy = _df_check.loc[_df_check["Visibility"] == 1, "Y"].tolist()
_W, _H = _PIL.open(FRAME_PATHS[0]).size

print(f"Frame size      : {_W} x {_H}")
print(f"CSV X range     : {min(_vx):.0f} .. {max(_vx):.0f}   ({100*(max(_vx)-min(_vx))/_W:.0f}% of width)")
print(f"CSV Y range     : {min(_vy):.0f} .. {max(_vy):.0f}   ({100*(max(_vy)-min(_vy))/_H:.0f}% of height)")

if max(_vx) > _W or max(_vy) > _H:
    print("\nWARNING: labels exceed frame bounds - check your CSV and frame files.")
elif (max(_vx) - min(_vx)) / _W < 0.40 and (max(_vy) - min(_vy)) / _H < 0.40:
    print("\nWARNING: labels cover less than 40% of the frame in both axes.")
    print(f"  If your frames are {_W}x{_H} but labels were annotated at a smaller size,")
    print(f"  set SCALE = {_W}/{int(round(max(_vx)))} = {_W/max(_vx):.2f} below and re-run.")
    print("  Then check the §8 overlay — the disc must sit on the ball.")
else:
    print("\nOK: coordinates span the frame as expected.")

# ---- apply: coordinates are already correct, no scaling needed -----------
SCALE = 1.0
if abs(SCALE - 1.0) > 1e-6:
    LABELS = {i: (x * SCALE, y * SCALE, v) for i, (x, y, v) in LABELS.items()}
    print(f"\nRescaled labels by {SCALE}.")
else:
    print("SCALE = 1.0 — labels used as-is from the CSV.")

# ---- quick visual spot-check ----------------------------------------------
import matplotlib.pyplot as plt, cv2 as _cv2
_vis_idx = [i for i, v in LABELS.items() if v[2]]
_pick    = _vis_idx[::max(1, len(_vis_idx) // 4)][:4]
fig, axes = plt.subplots(1, len(_pick), figsize=(4.5 * len(_pick), 3.4))
for ax, i in zip(np.atleast_1d(axes), _pick):
    im = _cv2.cvtColor(_cv2.imread(FRAME_PATHS[i]), _cv2.COLOR_BGR2RGB)
    x, y, _ = LABELS[i]
    ax.imshow(im)
    ax.add_patch(plt.Circle((x, y), max(_W, _H) * 0.018, fill=False,
                             color="red", lw=2))
    ax.plot(x, y, "r+", ms=12)
    ax.set_title(f"frame {i}  ({x:.0f}, {y:.0f})", fontsize=9)
    ax.axis("off")
plt.suptitle("Red ring MUST sit on the ball.\n"
             "If it is in the wrong place, set SCALE above and re-run.", fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
import os, sys, re, glob, json, time, math, random, shutil, textwrap, pathlib
import os.path as osp
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu  ", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Settings -> Accelerator -> GPU. Training will be unusably slow.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WORK    = "/kaggle/working" if osp.isdir("/kaggle/working") else "./work"
INPUT   = "/kaggle/input"   if osp.isdir("/kaggle/input")   else "./input"
WEIGHTS = f"{WORK}/weights"
DATA    = f"{WORK}/pickleball"
RUNS    = f"{WORK}/runs"
for d in (WEIGHTS, DATA, RUNS):
    os.makedirs(d, exist_ok=True)

def set_seed(seed=1234):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed()
print("\nwork:", WORK)

In [ ]:
# ---------------------------------------------------------------------------
# Mirrors configs/model/wasb.yaml + dataloader/default.yaml + loss/hm_qfl.yaml
# ---------------------------------------------------------------------------
CFG = {
    "model": {
        "name": "hrnet",
        "frames_in": 3,          # 3 frames in  -> 9 input channels
        "frames_out": 3,         # 3 heatmaps out (MIMO)
        "inp_height": 288, "inp_width": 512,
        "out_height": 288, "out_width": 512,
        "rgb_diff": False,
        "out_scales": [0],
        "MODEL": {
            "EXTRA": {
                "FINAL_CONV_KERNEL": 1,
                "PRETRAINED_LAYERS": ["*"],
                "STEM":   {"INPLANES": 64, "STRIDES": [1, 1]},
                "STAGE1": {"NUM_MODULES": 1, "NUM_BRANCHES": 1, "BLOCK": "BOTTLENECK",
                           "NUM_BLOCKS": [1], "NUM_CHANNELS": [32], "FUSE_METHOD": "SUM"},
                "STAGE2": {"NUM_MODULES": 1, "NUM_BRANCHES": 2, "BLOCK": "BASIC",
                           "NUM_BLOCKS": [2, 2], "NUM_CHANNELS": [16, 32], "FUSE_METHOD": "SUM"},
                "STAGE3": {"NUM_MODULES": 1, "NUM_BRANCHES": 3, "BLOCK": "BASIC",
                           "NUM_BLOCKS": [2, 2, 2], "NUM_CHANNELS": [16, 32, 64],
                           "FUSE_METHOD": "SUM"},
                "STAGE4": {"NUM_MODULES": 1, "NUM_BRANCHES": 4, "BLOCK": "BASIC",
                           "NUM_BLOCKS": [2, 2, 2, 2], "NUM_CHANNELS": [16, 32, 64, 128],
                           "FUSE_METHOD": "SUM"},
                "DECONV": {"NUM_DECONVS": 0, "KERNEL_SIZE": [], "NUM_BASIC_BLOCKS": 2},
            },
            "INIT_WEIGHTS": True,
        },
    },
    # dataloader/default.yaml  -- NOTE: 'sigmas' is a DISC RADIUS in pixels, not a Gaussian sigma
    "dataloader": {
        "heatmap": {"name": "binary_fixed_size", "sigmas": [2.5],
                    "mags": [1.0], "min_value": 0.7},
    },
    # loss/hm_qfl.yaml -- the loss WASB was actually trained with
    "loss": {"name": "heatmap", "sub_name": "quality_focal",
             "beta": 2.0, "auto_weight": False, "scales": [0]},
    # paper recipe: Adam, 3e-4, 30 epochs
    "optimizer": {"name": "adam", "learning_rate": 3e-4, "weight_decay": 0.0},
    # transform/full.yaml
    "transform": {
        "train": {"color_jitter": {"p": 0.5, "brightness": 0.1, "contrast": 0.1,
                                   "saturation": 0.1, "hue": 0.1},
                  "horizontal_flip": {"p": 0.5},
                  "crop": {"p": 0.5, "max_rescale": 0.125}},
        "test":  {"color_jitter": {"p": 0.0, "brightness": 0.0, "contrast": 0.0,
                                   "saturation": 0.0, "hue": 0.0},
                  "horizontal_flip": {"p": 0.0},
                  "crop": {"p": 0.0, "max_rescale": 0.125}},
    },
    "detector": {"postprocessor": {"score_threshold": 0.5, "scales": [0],
                                   "blob_det_method": "concomp", "use_hm_weight": True}},
    "runner": {"eval": {"dist_threshold": 4}},
}

INP_W, INP_H = CFG["model"]["inp_width"],  CFG["model"]["inp_height"]
OUT_W, OUT_H = CFG["model"]["out_width"],  CFG["model"]["out_height"]
FRAMES_IN    = CFG["model"]["frames_in"]
FRAMES_OUT   = CFG["model"]["frames_out"]
SCALE0       = 0

print(f"input  {FRAMES_IN} frames -> {3*FRAMES_IN} ch, {INP_H}x{INP_W}")
print(f"output {FRAMES_OUT} heatmaps, {OUT_H}x{OUT_W}")
print(f"target disc radius: {CFG['dataloader']['heatmap']['sigmas'][0]} px")
print(f"loss: {CFG['loss']['sub_name']} (beta={CFG['loss']['beta']})")

---
## 2. HRNet, verbatim

Copied unmodified from `src/models/hrnet.py` (originally Microsoft's HRNet, MIT licensed, as
adapted by the WASB authors). Verbatim is the point: any rename or restructure breaks the
strict `load_state_dict` in §3 and you would silently train from partially random weights.

Two things worth noticing in the architecture:

- **The stem does not downsample.** `STEM.STRIDES: [1, 1]`, so the output heatmap is at full
  512×288 input resolution. That is the "high-resolution feature extraction" of the paper's
  title, and it is why this model localises a 5-pixel ball where a stride-4 detector cannot.
- **`forward` returns a dict** keyed by output scale, `{0: tensor}`, not a bare tensor.

In [ ]:
# ==========================================================================
# VERBATIM from nttcom/WASB-SBDT  src/models/hrnet.py
# (c) Microsoft, MIT License. Written by Bin Xiao, modified by Bowen Cheng.
# Do not restructure - the tennis checkpoint is keyed to these exact names.
# ==========================================================================
import os
import logging

import torch
import torch.nn as nn


BN_MOMENTUM = 0.1
logger = logging.getLogger(__name__)


def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes, momentum=BN_MOMENTUM)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes, momentum=BN_MOMENTUM)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes, momentum=BN_MOMENTUM)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes, momentum=BN_MOMENTUM)
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, kernel_size=1,
                               bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion,
                                  momentum=BN_MOMENTUM)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class HighResolutionModule(nn.Module):
    def __init__(self, num_branches, blocks, num_blocks, num_inchannels,
                 num_channels, fuse_method, multi_scale_output=True):
        super(HighResolutionModule, self).__init__()
        self._check_branches(
            num_branches, blocks, num_blocks, num_inchannels, num_channels)

        self.num_inchannels = num_inchannels
        self.fuse_method = fuse_method
        self.num_branches = num_branches

        self.multi_scale_output = multi_scale_output

        self.branches = self._make_branches(
            num_branches, blocks, num_blocks, num_channels)
        self.fuse_layers = self._make_fuse_layers()
        self.relu = nn.ReLU(True)

    def _check_branches(self, num_branches, blocks, num_blocks,
                        num_inchannels, num_channels):
        if num_branches != len(num_blocks):
            error_msg = 'NUM_BRANCHES({}) <> NUM_BLOCKS({})'.format(
                num_branches, len(num_blocks))
            logger.error(error_msg)
            raise ValueError(error_msg)

        if num_branches != len(num_channels):
            error_msg = 'NUM_BRANCHES({}) <> NUM_CHANNELS({})'.format(
                num_branches, len(num_channels))
            logger.error(error_msg)
            raise ValueError(error_msg)

        if num_branches != len(num_inchannels):
            error_msg = 'NUM_BRANCHES({}) <> NUM_INCHANNELS({})'.format(
                num_branches, len(num_inchannels))
            logger.error(error_msg)
            raise ValueError(error_msg)

    def _make_one_branch(self, branch_index, block, num_blocks, num_channels,
                         stride=1):
        downsample = None
        if stride != 1 or \
           self.num_inchannels[branch_index] != num_channels[branch_index] * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.num_inchannels[branch_index],
                          num_channels[branch_index] * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(num_channels[branch_index] * block.expansion,
                               momentum=BN_MOMENTUM),
            )

        layers = []
        layers.append(block(self.num_inchannels[branch_index],
                            num_channels[branch_index], stride, downsample))
        self.num_inchannels[branch_index] = \
            num_channels[branch_index] * block.expansion
        for i in range(1, num_blocks[branch_index]):
            layers.append(block(self.num_inchannels[branch_index],
                                num_channels[branch_index]))

        return nn.Sequential(*layers)

    def _make_branches(self, num_branches, block, num_blocks, num_channels):
        branches = []

        for i in range(num_branches):
            branches.append(
                self._make_one_branch(i, block, num_blocks, num_channels))

        return nn.ModuleList(branches)

    def _make_fuse_layers(self):
        if self.num_branches == 1:
            return None

        num_branches = self.num_branches
        num_inchannels = self.num_inchannels
        fuse_layers = []
        for i in range(num_branches if self.multi_scale_output else 1):
            fuse_layer = []
            for j in range(num_branches):
                if j > i:
                    fuse_layer.append(nn.Sequential(
                        nn.Conv2d(num_inchannels[j],
                                  num_inchannels[i],
                                  1,
                                  1,
                                  0,
                                  bias=False),
                        nn.BatchNorm2d(num_inchannels[i]),
                        nn.Upsample(scale_factor=2**(j-i), mode='nearest')))
                elif j == i:
                    fuse_layer.append(None)
                else:
                    conv3x3s = []
                    for k in range(i-j):
                        if k == i - j - 1:
                            num_outchannels_conv3x3 = num_inchannels[i]
                            conv3x3s.append(nn.Sequential(
                                nn.Conv2d(num_inchannels[j],
                                          num_outchannels_conv3x3,
                                          3, 2, 1, bias=False),
                                nn.BatchNorm2d(num_outchannels_conv3x3)))
                        else:
                            num_outchannels_conv3x3 = num_inchannels[j]
                            conv3x3s.append(nn.Sequential(
                                nn.Conv2d(num_inchannels[j],
                                          num_outchannels_conv3x3,
                                          3, 2, 1, bias=False),
                                nn.BatchNorm2d(num_outchannels_conv3x3),
                                nn.ReLU(True)))
                    fuse_layer.append(nn.Sequential(*conv3x3s))
            fuse_layers.append(nn.ModuleList(fuse_layer))

        return nn.ModuleList(fuse_layers)

    def get_num_inchannels(self):
        return self.num_inchannels

    def forward(self, x):
        if self.num_branches == 1:
            return [self.branches[0](x[0])]

        for i in range(self.num_branches):
            x[i] = self.branches[i](x[i])

        x_fuse = []

        for i in range(len(self.fuse_layers)):
            y = x[0] if i == 0 else self.fuse_layers[i][0](x[0])
            for j in range(1, self.num_branches):
                if i == j:
                    y = y + x[j]
                else:
                    y = y + self.fuse_layers[i][j](x[j])
            x_fuse.append(self.relu(y))

        return x_fuse


blocks_dict = {
    'BASIC': BasicBlock,
    'BOTTLENECK': Bottleneck
}


# https://github.com/HRNet/HRNet-Image-Classification/blob/master/lib/models/cls_hrnet.py
class HRNet(nn.Module):

    def __init__(self, cfg, **kwargs):
        super(HRNet, self).__init__()

        self._frames_in  = cfg['frames_in']
        self._frames_out = cfg['frames_out']
        self._out_scales = cfg['out_scales']
        self._stem_strides  = cfg['MODEL']['EXTRA']['STEM']['STRIDES']
        self._stem_inplanes = cfg['MODEL']['EXTRA']['STEM']['INPLANES']

        self.conv1 = nn.Conv2d(3*self._frames_in, self._stem_inplanes, kernel_size=3, stride=self._stem_strides[0], padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(self._stem_inplanes, momentum=BN_MOMENTUM)
        self.conv2 = nn.Conv2d(self._stem_inplanes, self._stem_inplanes, kernel_size=3, stride=self._stem_strides[1], padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(self._stem_inplanes, momentum=BN_MOMENTUM)
        self.relu  = nn.ReLU(inplace=True)

        self.stage1_cfg = cfg['MODEL']['EXTRA']['STAGE1']
        num_channels    = self.stage1_cfg['NUM_CHANNELS'][0]
        block = blocks_dict[self.stage1_cfg['BLOCK']]
        num_blocks = self.stage1_cfg['NUM_BLOCKS'][0]
        self.layer1 = self._make_layer(block, self._stem_inplanes, num_channels, num_blocks)
        stage1_out_channel = block.expansion*num_channels

        self.stage2_cfg = cfg['MODEL']['EXTRA']['STAGE2']
        num_channels = self.stage2_cfg['NUM_CHANNELS']
        block = blocks_dict[self.stage2_cfg['BLOCK']]
        num_channels = [
            num_channels[i] * block.expansion for i in range(len(num_channels))]
        self.transition1 = self._make_transition_layer(
            [stage1_out_channel], num_channels)
        self.stage2, pre_stage_channels = self._make_stage(
            self.stage2_cfg, num_channels)

        self.stage3_cfg = cfg['MODEL']['EXTRA']['STAGE3']
        num_channels = self.stage3_cfg['NUM_CHANNELS']
        block = blocks_dict[self.stage3_cfg['BLOCK']]
        num_channels = [
            num_channels[i] * block.expansion for i in range(len(num_channels))]
        self.transition2 = self._make_transition_layer(
            pre_stage_channels, num_channels)
        self.stage3, pre_stage_channels = self._make_stage(
            self.stage3_cfg, num_channels)

        self.stage4_cfg = cfg['MODEL']['EXTRA']['STAGE4']
        num_channels = self.stage4_cfg['NUM_CHANNELS']
        block = blocks_dict[self.stage4_cfg['BLOCK']]
        num_channels = [
            num_channels[i] * block.expansion for i in range(len(num_channels))]
        self.transition3 = self._make_transition_layer(
            pre_stage_channels, num_channels)
        self.stage4, pre_stage_channels = self._make_stage(self.stage4_cfg, num_channels, multi_scale_output=True)

        self.num_deconvs       = cfg['MODEL']['EXTRA']['DECONV']['NUM_DECONVS']
        self.deconv_config     = cfg['MODEL']['EXTRA']['DECONV']
        self.pretrained_layers = cfg['MODEL']['EXTRA']['PRETRAINED_LAYERS']

        self.deconv_layers = self._make_deconv_layers(cfg, pre_stage_channels[0])
        self.final_layers  = self._make_final_layers(cfg, pre_stage_channels)

    def _get_deconv_cfg(self, deconv_kernel):
        if deconv_kernel == 4:
            padding = 1
            output_padding = 0
        elif deconv_kernel == 3:
            padding = 1
            output_padding = 1
        elif deconv_kernel == 2:
            padding = 0
            output_padding = 0

        return deconv_kernel, padding, output_padding

    def _make_final_layers(self, cfg, channels):
        kernel_size = cfg['MODEL']['EXTRA']['FINAL_CONV_KERNEL']
        layers      = []
        for scale in self._out_scales:
            layers.append( nn.Conv2d(in_channels=channels[scale], out_channels=self._frames_out, kernel_size=kernel_size) )
        return nn.ModuleList(layers)

    def _make_deconv_layers(self, cfg, input_channels):
        extra      = cfg.MODEL.EXTRA
        deconv_cfg = extra.DECONV

        deconv_layers = []
        for i in range(deconv_cfg.NUM_DECONVS): 
            output_channels                        = input_channels
            deconv_kernel, padding, output_padding = self._get_deconv_cfg(deconv_cfg.KERNEL_SIZE[i])

            layers = []
            layers.append(nn.Sequential(
                nn.ConvTranspose2d(
                    in_channels=input_channels,
                    out_channels=output_channels,
                    kernel_size=deconv_kernel,
                    stride=2,
                    padding=padding,
                    output_padding=output_padding,
                    bias=False),
                nn.BatchNorm2d(output_channels, momentum=BN_MOMENTUM),
                nn.ReLU(inplace=True)
            ))
            deconv_layers.append(nn.Sequential(*layers))
            input_channels = output_channels

        return nn.ModuleList(deconv_layers)

    def _make_transition_layer(
            self, num_channels_pre_layer, num_channels_cur_layer):
        num_branches_cur = len(num_channels_cur_layer)
        num_branches_pre = len(num_channels_pre_layer)

        transition_layers = []
        for i in range(num_branches_cur):
            if i < num_branches_pre:
                if num_channels_cur_layer[i] != num_channels_pre_layer[i]:
                    transition_layers.append(nn.Sequential(
                        nn.Conv2d(num_channels_pre_layer[i],
                                  num_channels_cur_layer[i],
                                  3,
                                  1,
                                  1,
                                  bias=False),
                        nn.BatchNorm2d(
                            num_channels_cur_layer[i], momentum=BN_MOMENTUM),
                        nn.ReLU(inplace=True)))
                else:
                    transition_layers.append(None)
            else:
                conv3x3s = []
                for j in range(i+1-num_branches_pre):
                    inchannels = num_channels_pre_layer[-1]
                    outchannels = num_channels_cur_layer[i] \
                        if j == i-num_branches_pre else inchannels
                    conv3x3s.append(nn.Sequential(
                        nn.Conv2d(
                            inchannels, outchannels, 3, 2, 1, bias=False),
                        nn.BatchNorm2d(outchannels, momentum=BN_MOMENTUM),
                        nn.ReLU(inplace=True)))
                transition_layers.append(nn.Sequential(*conv3x3s))

        return nn.ModuleList(transition_layers)

    def _make_layer(self, block, inplanes, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion, momentum=BN_MOMENTUM),
            )

        layers = []
        layers.append(block(inplanes, planes, stride, downsample))
        inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(inplanes, planes))

        return nn.Sequential(*layers)

    def _make_stage(self, layer_config, num_inchannels,
                    multi_scale_output=True):
        num_modules = layer_config['NUM_MODULES']
        num_branches = layer_config['NUM_BRANCHES']
        num_blocks = layer_config['NUM_BLOCKS']
        num_channels = layer_config['NUM_CHANNELS']
        block = blocks_dict[layer_config['BLOCK']]
        fuse_method = layer_config['FUSE_METHOD']

        modules = []
        for i in range(num_modules):
            # multi_scale_output is only used last module
            if not multi_scale_output and i == num_modules - 1:
                reset_multi_scale_output = False
            else:
                reset_multi_scale_output = True

            modules.append(
                HighResolutionModule(num_branches,
                                      block,
                                      num_blocks,
                                      num_inchannels,
                                      num_channels,
                                      fuse_method,
                                      reset_multi_scale_output)
            )
            num_inchannels = modules[-1].get_num_inchannels()

        return nn.Sequential(*modules), num_inchannels

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.layer1(x)

        x_list = []
        for i in range(self.stage2_cfg['NUM_BRANCHES']):
            if self.transition1[i] is not None:
                x_list.append(self.transition1[i](x))
            else:
                x_list.append(x)
        y_list = self.stage2(x_list)

        x_list = []
        for i in range(self.stage3_cfg['NUM_BRANCHES']):
            if self.transition2[i] is not None:
                x_list.append(self.transition2[i](y_list[-1]))
            else:
                x_list.append(y_list[i])
        y_list = self.stage3(x_list)

        x_list = []
        for i in range(self.stage4_cfg['NUM_BRANCHES']):
            if self.transition3[i] is not None:
                x_list.append(self.transition3[i](y_list[-1]))
            else:
                x_list.append(y_list[i])
        y_list = self.stage4(x_list)

        y_out = {}
        for scale in self._out_scales:
            x = y_list[scale]
            for i in range(self.num_deconvs):
                x = self.deconv_layers[i][scale](x)
            y = self.final_layers[scale](x)
            y_out[scale] = y
        return y_out

    def init_weights(self, pretrained='',):
        logger.info('=> init weights from normal distribution')
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        if os.path.isfile(pretrained):
            pretrained_dict = torch.load(pretrained)
            logger.info('=> loading pretrained model {}'.format(pretrained))
            model_dict = self.state_dict()
            pretrained_dict = {k: v for k, v in pretrained_dict.items()
                               if k in model_dict.keys()}
            for k, _ in pretrained_dict.items():
                logger.info(
                    '=> loading {} pretrained model {}'.format(k, pretrained))
            model_dict.update(pretrained_dict)
            self.load_state_dict(model_dict)

In [ ]:
class AttrDict(dict):
    '''dict that also supports attribute access.

    The repo hands HRNet an OmegaConf DictConfig, which allows BOTH cfg['MODEL'] and
    cfg.MODEL. Most of hrnet.py uses the subscript form, but _make_deconv_layers uses
    the attribute form (cfg.MODEL.EXTRA, deconv_cfg.NUM_DECONVS). Passing a plain dict
    therefore raises AttributeError inside the constructor. Rather than edit the model
    source - which would put the verbatim guarantee at risk - adapt the config.
    '''
    def __init__(self, d=None):
        super().__init__()
        for k, v in (d or {}).items():
            self[k] = AttrDict(v) if isinstance(v, dict) else v
    def __getattr__(self, k):
        try:
            return self[k]
        except KeyError:
            raise AttributeError(k)
    def __setattr__(self, k, v):
        self[k] = v


def build_model(cfg):
    '''Mirrors models/__init__.py build_model() for the hrnet/WASB branch.'''
    assert cfg["model"]["name"] == "hrnet"
    return HRNet(AttrDict(cfg["model"]))

model = build_model(CFG)
n_all   = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"WASB / HRNet: {n_all:,} params ({n_train:,} trainable)")

with torch.no_grad():
    y = model(torch.randn(2, 3 * FRAMES_IN, INP_H, INP_W))
print("forward output is a dict:", type(y).__name__, "| keys", list(y.keys()))
print("scale 0 logits:", tuple(y[0].shape), "-> (B, frames_out, H, W)")
assert y[0].shape[1] == FRAMES_OUT and y[0].shape[2:] == (OUT_H, OUT_W)
print("\nheatmap is at FULL input resolution - no downsampling in the stem")

---
## 3. Tennis weights, strict load

From the repo's model zoo. Checkpoints are dicts with a `model_state_dict` key; weights saved
under `DataParallel` carry a `module.` prefix that has to be stripped first.

| Sport | Google Drive ID |
|---|---|
| **Tennis** | `14AeyIOCQ2UaQmbZLNQJa1H_eSwxUXk7z` |
| Badminton | `17Ac0pO5oryh1JwgwTFQTjOKHY3umbDQu` |
| Soccer | `1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l` |

Tennis is the natural start for pickleball: outdoor court, comparable camera framing, similar
ball-to-frame scale. Badminton saw roughly 3× more annotated frames and a harder small-object
target, so it is worth a 3-epoch bake-off once the pipeline runs.

**No internet?** Download `wasb_tennis_best.pth.tar` locally, attach it as a Kaggle Dataset —
the cell checks `/kaggle/input` first.

The `strict=True` load is the real test that §2 is faithful. If it passes, every one of the
368 tensors found its home.

In [ ]:
WASB_TENNIS_ID = "14AeyIOCQ2UaQmbZLNQJa1H_eSwxUXk7z"
TENNIS_CKPT    = f"{WEIGHTS}/wasb_tennis_best.pth.tar"

hits = glob.glob(f"{INPUT}/**/wasb_tennis*.pth.tar", recursive=True)
if hits:
    TENNIS_CKPT = hits[0]
    print("using attached dataset:", TENNIS_CKPT)
elif not osp.exists(TENNIS_CKPT):
    try:
        import gdown
    except ImportError:
        get_ipython().system('pip install -q gdown')
        import gdown
    gdown.download(id=WASB_TENNIS_ID, output=TENNIS_CKPT, quiet=False)

assert osp.exists(TENNIS_CKPT), (
    "No checkpoint. Enable Internet, or upload wasb_tennis_best.pth.tar as a Kaggle Dataset.")
print(f"{TENNIS_CKPT}  ({os.path.getsize(TENNIS_CKPT)/1e6:.1f} MB)")


def load_state_dict_clean(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck["model_state_dict"] if "model_state_dict" in ck else ck
    return {k[len("module."):] if k.startswith("module.") else k: v for k, v in sd.items()}


sd = load_state_dict_clean(TENNIS_CKPT)
print(f"\ncheckpoint: {len(sd)} tensors, {sum(v.numel() for v in sd.values()):,} params")

missing, unexpected = [], []
try:
    model.load_state_dict(sd)                       # strict=True
    print("\nSTRICT LOAD OK - inlined HRNet matches the published tennis weights exactly")
except RuntimeError as e:
    print("\nSTRICT LOAD FAILED:\n", str(e)[:1500]); raise

model = model.to(DEVICE)
torch.save({"model_state_dict": sd}, f"{WEIGHTS}/wasb_tennis_clean.pth.tar")
INIT_CKPT = f"{WEIGHTS}/wasb_tennis_clean.pth.tar"

---
## 4. Input warp and target heatmaps

Two details from `dataloaders/dataset_loader.py` and `utils/` that are easy to get wrong.

**The input warp is an affine transform, not a resize.** The source box is a *square* of side
`max(H, W)` centred on the frame, mapped to a square of side `out_w` centred on the output
canvas. For 16:9 footage into 512×288 this works out to an exact aspect-preserving resize; for
any other aspect ratio it letterboxes instead of stretching. Substituting `cv2.resize` would
distort non-16:9 input and quietly mismatch what the tennis weights expect. The inverse
transform is what maps detections back to original pixel coordinates.

**`sigmas: [2.5]` is a hard disc radius, not a Gaussian sigma.** `gen_binary_map` fills every
pixel within `r` of the labelled centre with 1.0 and leaves the rest 0.0 — a flat disc about
5 px across. This is worth re-tuning for pickleball: the target should roughly cover the ball's
apparent size. Too small and the positive class nearly vanishes; too large and localisation
goes mushy. §8 measures it.

(`gen_heatmap`, the soft variant, is used by the repo only for hard-negative frames. Included
for completeness.)

In [ ]:
# ---- VERBATIM from utils/image.py (Microsoft / Xingyi Zhou, MIT) ----------
def get_dir(src_point, rot_rad):
    sn, cs = np.sin(rot_rad), np.cos(rot_rad)
    return [src_point[0] * cs - src_point[1] * sn,
            src_point[0] * sn + src_point[1] * cs]

def get_3rd_point(a, b):
    direct = a - b
    return b + np.array([-direct[1], direct[0]], dtype=np.float32)

def get_affine_transform(center, scale, rot, output_size,
                         shift=np.array([0, 0], dtype=np.float32), inv=0):
    if not isinstance(scale, np.ndarray) and not isinstance(scale, list):
        scale = np.array([scale, scale], dtype=np.float32)
    scale_tmp = scale
    src_w = scale_tmp[0]
    dst_w, dst_h = output_size[0], output_size[1]

    rot_rad = np.pi * rot / 180
    src_dir = get_dir([0, src_w * -0.5], rot_rad)
    dst_dir = np.array([0, dst_w * -0.5], np.float32)

    src = np.zeros((3, 2), dtype=np.float32)
    dst = np.zeros((3, 2), dtype=np.float32)
    src[0, :] = center + scale_tmp * shift
    src[1, :] = center + src_dir + scale_tmp * shift
    dst[0, :] = [dst_w * 0.5, dst_h * 0.5]
    dst[1, :] = np.array([dst_w * 0.5, dst_h * 0.5], np.float32) + dst_dir
    src[2:, :] = get_3rd_point(src[0, :], src[1, :])
    dst[2:, :] = get_3rd_point(dst[0, :], dst[1, :])

    if inv:
        return cv2.getAffineTransform(np.float32(dst), np.float32(src))
    return cv2.getAffineTransform(np.float32(src), np.float32(dst))

def affine_transform(pt, t):
    new_pt = np.array([pt[0], pt[1], 1.], dtype=np.float32).T
    return np.dot(t, new_pt)[:2]

# ---- VERBATIM from dataloaders/dataset_loader.py -------------------------
def get_transform(img, input_wh, inv=0):
    h, w, _ = img.shape
    c = np.array([w / 2., h / 2.], dtype=np.float32)
    s = max(h, w) * 1.0
    return get_affine_transform(c, s, 0, [input_wh[0], input_wh[1]], inv=inv)

# ---- VERBATIM from utils/heatmap.py --------------------------------------
def gen_binary_map(wh, cxy, r, data_type=np.float32):
    w, h = wh; cx, cy = cxy
    if cx < 0 or cy < 0:
        return np.zeros((h, w), dtype=data_type)
    x, y = np.meshgrid(np.linspace(1, w, w), np.linspace(1, h, h))
    distmap = ((y - (cy + 1)) ** 2) + ((x - (cx + 1)) ** 2)
    bmap = np.zeros_like(distmap)
    bmap[distmap <= r ** 2] = 1
    return bmap.astype(data_type)

def gen_heatmap(wh, cxy, r, data_type=np.float32, min_value=0.7):
    w, h = wh; cx, cy = cxy
    if cx < 0 or cy < 0:
        return np.zeros((h, w), dtype=data_type)
    x, y = np.meshgrid(np.linspace(1, w, w), np.linspace(1, h, h))
    distmap = ((y - (cy + 1)) ** 2) + ((x - (cx + 1)) ** 2)
    heatmap = np.exp(-distmap / (r ** 2)) / np.exp(-1.) * min_value
    heatmap[heatmap < 0.5] = 0
    heatmap[heatmap > 1] = 1.
    return heatmap.astype(data_type)


class BinaryFixedSizeMapGenerator:
    def __init__(self, cfg):
        self._sigma = cfg["sigmas"][0]
        self._min_value = cfg["min_value"]
        self._data_type = np.float32
    def __call__(self, wh, cxy, binary=True):
        if binary:
            return gen_binary_map(wh, cxy, self._sigma, self._data_type)
        return gen_heatmap(wh, cxy, self._sigma, self._data_type, min_value=self._min_value)


# --- demo: 1280x720 in, 512x288 out, ball at (640, 360) -------------------
probe = np.zeros((720, 1280, 3), np.uint8)
t_in  = get_transform(probe, (INP_W, INP_H))
t_out = get_transform(probe, (OUT_W, OUT_H))
t_inv = get_transform(probe, (OUT_W, OUT_H), inv=1)

for px, py in [(640, 360), (100, 100), (1180, 620)]:
    m = affine_transform(np.array([px, py], np.float32), t_out)
    back = affine_transform(m.astype(np.float32), t_inv)
    print(f"  ({px:4d},{py:4d}) -> warped ({m[0]:6.1f},{m[1]:6.1f}) -> back "
          f"({back[0]:6.1f},{back[1]:6.1f})")

hm_gen = BinaryFixedSizeMapGenerator(CFG["dataloader"]["heatmap"])
hm = hm_gen((OUT_W, OUT_H), (256, 144))
print(f"\ntarget: {hm.shape}, positives {int(hm.sum())} px "
      f"({100*hm.mean():.4f}% of the map) - severe class imbalance, which is what the "
      f"focal-style loss is for")

---
## 5. Quality focal loss

The loss WASB was trained with. From `losses/quality_focal_loss.py`, wrapped by
`losses/heatmap.py`, which applies a **clamped** sigmoid to the logits first:

```python
def _sigmoid(x):
    return torch.clamp(x.sigmoid_(), min=1e-4, max=1-1e-4)
```

The clamp is load-bearing, not decoration: the loss takes `log(p)` and `log(1-p)` directly, so
an unclamped sigmoid saturating to exactly 0 or 1 produces `inf` and the run dies with NaN
within an epoch or two.

Quality focal loss, for prediction `p` and target `q`:

$$\mathcal{L} = -\,|p - q|^{\beta}\,\big[(1-q)\log(1-p) + q\log p\big]$$

The `|p − q|^β` factor is the focal modulation: confidently-correct pixels (almost all of the
147k background pixels per map) contribute almost nothing, so the ~20 ball pixels dominate the
gradient. Unlike standard focal loss it handles *continuous* targets in [0,1], which is what
"position-aware training" in the paper refers to.

WBCE is included below for comparison only — it is TrackNetV2's loss, and the repo's own
comment notes it is essentially focal loss with γ=2.

In [ ]:
def _sigmoid(x):
    '''VERBATIM from utils/utils.py - the clamp prevents log(0) -> NaN.'''
    return torch.clamp(x.sigmoid_(), min=1e-4, max=1 - 1e-4)


class QualityFocalLoss(nn.Module):
    '''VERBATIM from losses/quality_focal_loss.py. https://arxiv.org/abs/2006.04388'''
    def __init__(self, beta=2.0, auto_weight=False, scales=[0]):
        super().__init__()
        self._beta = beta
        self._auto_weight = auto_weight
        self._ws = {}
        for scale in scales:
            self._ws[f"loss_w_s{scale}"] = (nn.Parameter(0. * torch.ones(1))
                                            if auto_weight else 1.0)
        if auto_weight:
            self._ws = nn.ParameterDict(self._ws)

    def forward(self, inputs, targets):
        loss_acc = 0
        for scale in inputs.keys():
            loss = (torch.abs(inputs[scale] - targets[scale]) ** self._beta
                    * ((1 - targets[scale]) * torch.log(1 - inputs[scale])
                       + targets[scale] * torch.log(inputs[scale])))
            loss = torch.mean(-loss)
            if self._auto_weight:
                w = self._ws[f"loss_w_s{scale}"]
                loss_acc += loss * torch.exp(-w) + w
            else:
                loss_acc += loss * self._ws[f"loss_w_s{scale}"]
        return loss_acc


class WBCELoss(nn.Module):
    '''VERBATIM from losses/wbce.py - TrackNetV2's loss, for comparison only.'''
    def __init__(self, auto_weight=False, scales=[0]):
        super().__init__()
        self._auto_weight = auto_weight
        self._ws = {}
        for scale in scales:
            self._ws[f"loss_w_s{scale}"] = (nn.Parameter(0. * torch.ones(1))
                                            if auto_weight else 1.0)
        if auto_weight:
            self._ws = nn.ParameterDict(self._ws)

    def forward(self, inputs, targets):
        loss_acc = 0
        for scale in inputs.keys():
            loss = (((1 - inputs[scale]) ** 2) * targets[scale] * torch.log(inputs[scale])
                    + (inputs[scale] ** 2) * (1 - targets[scale])
                    * torch.log(1 - inputs[scale]))
            loss = torch.mean(-loss)
            loss_acc += loss * self._ws[f"loss_w_s{scale}"]
        return loss_acc


class HeatmapLoss(nn.Module):
    '''VERBATIM structure from losses/heatmap.py - sigmoid is applied HERE, not in the model.'''
    def __init__(self, cfg):
        super().__init__()
        sub = cfg["loss"]["sub_name"]
        if sub == "quality_focal":
            self._loss = QualityFocalLoss(beta=cfg["loss"]["beta"],
                                          auto_weight=cfg["loss"]["auto_weight"],
                                          scales=cfg["loss"]["scales"])
        elif sub == "wbce":
            self._loss = WBCELoss(auto_weight=cfg["loss"]["auto_weight"],
                                  scales=cfg["loss"]["scales"])
        elif sub == "mse":
            self._loss = nn.MSELoss()
        else:
            raise KeyError(f"invalid loss: {sub}")

    def forward(self, inputs, targets):
        return self._loss({s: _sigmoid(v) for s, v in inputs.items()}, targets)


criterion = HeatmapLoss(CFG).to(DEVICE)
print("loss:", CFG["loss"]["sub_name"], "| beta", CFG["loss"]["beta"])

# sanity: a near-perfect prediction must score far below a confidently wrong one
tgt = torch.from_numpy(hm_gen((OUT_W, OUT_H), (256, 144))).float()[None, None].to(DEVICE)
good = torch.where(tgt > 0.5,  6.0, -6.0)
bad  = torch.where(tgt > 0.5, -6.0,  6.0)
zero = torch.zeros_like(tgt)
with torch.no_grad():
    lg = criterion({0: good.clone()}, {0: tgt}).item()
    lb = criterion({0: bad.clone()},  {0: tgt}).item()
    lz = criterion({0: zero.clone()}, {0: tgt}).item()
print(f"  near-perfect {lg:.6f} | all-zeros logits {lz:.6f} | inverted {lb:.6f}")
assert lg < lz < lb and math.isfinite(lb), "loss is not behaving - do not train"
print("  ordering correct and finite")

---
## 6. Your data — one flat clip

Your Kaggle inputs are two separate datasets, a CSV and a frames folder:

```
/kaggle/input/csv111/2_1_fixed.csv
/kaggle/input/image_dataset/2_1_frames/...
```

That is **one clip**, not the match/clip tree the repo assumes, so §6–§7 are rewritten for it.
Frames are read straight from `/kaggle/input` — nothing is copied, which keeps you clear of the
working-directory quota.

### Read this before trusting any number this notebook prints

With a single clip there is **no clean train/test split**. Frames inside one rally are
near-duplicates: same court, same lighting, same camera, ball positions a few pixels apart.
Whatever you hold out, the model has effectively already seen.

The split below is the least-bad option — the first 75% of frames train, the last 25% test,
with a gap so no 3-frame window straddles the boundary. That prevents the grossest leak, a
window half in each split. It **cannot** prevent the real one.

Expect the F1 in §10 to be optimistic, possibly by a lot. To get a number you can act on,
record a second clip on a different day or court and hold it out entirely. Until then, treat
§10 as a smoke test that the pipeline learns *something*, not as a measure of how it will do on
new footage.

### Column detection

I don't know your CSV's schema, so the loader detects it. It handles `Frame/Visibility/X/Y`,
TrackNet-style `file name/visibility/x-coordinate/y-coordinate/status`, bare `frame,x,y` and
variants; aligns rows to frames by **filename** when the frame column holds names and by
**index** otherwise; and infers visibility from `NaN` or `(0,0)` coordinates when there is no
visibility column. Tested against 5 schemas × 4 filename patterns.

Run the inspection cell first and check its output against what you know about your data.

In [ ]:
# ---------------------------------------------------------------------------
# Locate the inputs. Set these by hand if auto-detection picks the wrong thing.
# ---------------------------------------------------------------------------
CSV_PATH   = None      # e.g. "/kaggle/input/csv111/2_1_fixed.csv"
FRAMES_DIR = None      # e.g. "/kaggle/input/image_dataset/2_1_frames"

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

if CSV_PATH is None:
    hits = sorted(glob.glob(f"{INPUT}/**/*.csv", recursive=True))
    print("CSV files found:")
    for h in hits:
        print("   ", h)
    assert hits, f"no CSV under {INPUT} - attach your dataset"
    CSV_PATH = hits[0]

if FRAMES_DIR is None:
    counts = {}
    for root, _, fnames in os.walk(INPUT):
        n = sum(1 for f in fnames if f.lower().endswith(IMG_EXT))
        if n:
            counts[root] = n
    print("\ndirectories containing images:")
    for d, n in sorted(counts.items(), key=lambda kv: -kv[1])[:8]:
        print(f"    {n:6d}  {d}")
    assert counts, f"no images under {INPUT}"
    FRAMES_DIR = max(counts, key=counts.get)

print(f"\nUSING\n  csv    : {CSV_PATH}\n  frames : {FRAMES_DIR}")

_df = pd.read_csv(CSV_PATH)
print(f"\nCSV shape {_df.shape}")
print("columns:", list(_df.columns))
print("\ndtypes:"); print(_df.dtypes.to_string())
print("\nhead:");   print(_df.head(8).to_string())
print("\ntail:");   print(_df.tail(3).to_string())

In [ ]:
def natural_key(s):
    # sort 2.jpg before 10.jpg
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", osp.basename(s))]

def list_frames(frames_dir):
    files = [p for p in glob.glob(osp.join(frames_dir, "**", "*"), recursive=True)
             if p.lower().endswith(IMG_EXT)]
    return sorted(files, key=natural_key)

# canonical name -> candidate column names, most specific FIRST
CANON = {
    "frame": ["frame", "file name", "filename", "file_name", "fname", "img", "image",
              "num", "frame_num", "frame_id", "frameid", "id", "index"],
    "visi":  ["visibility", "visible", "visi", "vis", "status"],
    "x":     ["x-coordinate", "x_coordinate", "xcoord", "ball_x", "x_pos", "cx", "x"],
    "y":     ["y-coordinate", "y_coordinate", "ycoord", "ball_y", "y_pos", "cy", "y"],
}

def detect_columns(df):
    norm = {c.strip().lower(): c for c in df.columns}
    out = {}
    for canon, cands in CANON.items():
        for cand in cands:
            if cand in norm and norm[cand] not in out.values():
                out[canon] = norm[cand]
                break
    return out


def load_flat_labels(csv_path, frames_dir, verbose=True):
    # CSV + frames folder -> (frame_paths, {idx: (x, y, visible)})
    df    = pd.read_csv(csv_path)
    files = list_frames(frames_dir)
    cols  = detect_columns(df)

    if verbose:
        print(f"CSV      : {osp.basename(csv_path)}  rows={len(df)}")
        print(f"  columns: {list(df.columns)}")
        print(f"  mapped : {cols}")
        print(f"Frames   : {len(files)} images")
        if files:
            print(f"  first {osp.basename(files[0])}   last {osp.basename(files[-1])}")

    if "x" not in cols or "y" not in cols:
        raise ValueError(f"no x/y columns detected in {list(df.columns)} - set them by hand")
    if not files:
        raise ValueError(f"no images under {frames_dir}")

    xs = pd.to_numeric(df[cols["x"]], errors="coerce").to_numpy(dtype=float)
    ys = pd.to_numeric(df[cols["y"]], errors="coerce").to_numpy(dtype=float)

    if "visi" in cols:
        vis = pd.to_numeric(df[cols["visi"]], errors="coerce").fillna(0).to_numpy() > 0
    else:
        vis = np.ones(len(df), dtype=bool)
        if verbose:
            print("  NOTE: no visibility column; inferring from NaN / non-positive coords")
    vis = vis & np.isfinite(xs) & np.isfinite(ys) & ~((xs <= 0) & (ys <= 0))
    xs, ys = np.nan_to_num(xs), np.nan_to_num(ys)

    frame_paths, labels = None, {}
    by_name = {osp.basename(p): p for p in files}

    # pandas >= 3.0 types string columns as 'str', not 'object', so an `== object`
    # test silently misses filename columns. Test for non-numeric instead.
    frame_is_text = ("frame" in cols
                     and not pd.api.types.is_numeric_dtype(df[cols["frame"]]))

    if frame_is_text:
        keys    = df[cols["frame"]].astype(str).str.strip().tolist()
        matched = [k for k in keys if k in by_name]
        if len(matched) < 0.5 * len(keys):                  # retry without extensions
            stem     = {osp.splitext(osp.basename(p))[0]: p for p in files}
            keys2    = [osp.splitext(osp.basename(k))[0] for k in keys]
            matched2 = [k for k in keys2 if k in stem]
            if len(matched2) > len(matched):
                keys, by_name, matched = keys2, stem, matched2
        if not matched:
            if verbose:
                print("  WARNING: frame column looks like filenames but none matched.")
                print(f"           csv e.g. {keys[:3]}")
                print(f"           dir e.g. {[osp.basename(p) for p in files[:3]]}")
                print("           falling back to POSITIONAL alignment - CHECK the "
                      "overlay in section 8 before training.")
            frame_is_text = False
        else:
            frame_paths = []
            for i, k in enumerate(keys):
                if k in by_name:
                    labels[len(frame_paths)] = (xs[i], ys[i], bool(vis[i]))
                    frame_paths.append(by_name[k])
            if verbose:
                print(f"  aligned by FILENAME: {len(frame_paths)}/{len(keys)} matched")

    if not frame_is_text:
        idx = None
        if "frame" in cols:
            cand = pd.to_numeric(df[cols["frame"]], errors="coerce")
            if cand.notna().mean() > 0.9:
                idx = cand.fillna(-1).astype(int).to_numpy()
                if idx.min() == 1 and idx.max() == len(files):
                    idx = idx - 1
                    if verbose:
                        print("  frame column is 1-based; shifted to 0-based")
        if idx is None:
            idx = np.arange(len(df))
            if verbose:
                print("  using CSV row order == sorted frame order (positional)")
        frame_paths = files
        n_ok = 0
        for r, i in enumerate(idx):
            if 0 <= i < len(files):
                labels[int(i)] = (xs[r], ys[r], bool(vis[r]))
                n_ok += 1
        if verbose:
            print(f"  aligned by INDEX: {n_ok}/{len(df)} rows onto {len(files)} frames")

    if verbose:
        nv = sum(1 for v in labels.values() if v[2])
        print(f"  labels: {len(labels)} total, {nv} visible "
              f"({100*nv/max(len(labels),1):.1f}%)")
        if nv:
            vx = [v[0] for v in labels.values() if v[2]]
            vy = [v[1] for v in labels.values() if v[2]]
            print(f"  x range {min(vx):.0f}..{max(vx):.0f}   "
                  f"y range {min(vy):.0f}..{max(vy):.0f}")
            W, H = Image.open(frame_paths[0]).size
            cov_x, cov_y = (max(vx) - min(vx)) / W, (max(vy) - min(vy)) / H
            print(f"  frame size {W}x{H}")
            print(f"  labels span {cov_x*100:.0f}% of width, {cov_y*100:.0f}% of height")
            if max(vx) > W or max(vy) > H:
                print("  FAIL: labels exceed frame bounds - wrong resolution, "
                      "or x/y swapped?")
            elif cov_x < 0.55 and cov_y < 0.55:
                print("  FAIL: labels sit in a small corner of the frame.")
                print(f"        implied annotation size ~{max(vx):.0f}x{max(vy):.0f}")
                print(f"        candidate scales: W {W/max(vx):.2f}x  H {H/max(vy):.2f}x")
                print("        -> section 6b will measure and correct this.")
            else:
                print("  OK: labels span the frame as expected")
    return frame_paths, labels


FRAME_PATHS, LABELS = load_flat_labels(CSV_PATH, FRAMES_DIR)
N_FRAMES = len(FRAME_PATHS)

---
## 6b. Coordinate check — using CSV values directly

Annotation tools often record coordinates on **downscaled** frames while the images on
disk stay at full resolution. When that happens every target disc lands in the top-left
corner, on empty court, and training quietly destroys the model: it learns to predict
nothing, anywhere.

It is invisible in the loss curve, which keeps falling the whole time — with ~147,000
background pixels and ~18 foreground, "predict nothing" already scores almost perfectly.

So this measures rather than assumes. For each candidate scale `s` it samples motion energy
at `(s·x, s·y)` across 30 frames, using `min(|f − f₋₁|, |f − f₊₁|)` — requiring change against
**both** neighbours, which suppresses players' trailing edges that would otherwise outscore
the ball. The true scale lands on a moving object and wins by a wide margin.

Tested against noise, JPEG artefacts and multiple moving distractors; correctly returns
`1.0` when labels are already right, so it is safe to leave in place.

**Check the four frames it draws.** The red ring must sit on the ball. If it doesn't, set
`SCALE` by hand — for 1920×1080 images annotated at 512×288 that is `1920/512 = 3.75`.


In [ ]:
# ---- confirm coordinates match the frame size ----------------------------
from PIL import Image as _PIL

_df_check = pd.read_csv(CSV_PATH)
_vx = _df_check.loc[_df_check["Visibility"] == 1, "X"].tolist()
_vy = _df_check.loc[_df_check["Visibility"] == 1, "Y"].tolist()
_W, _H = Image.open(FRAME_PATHS[0]).size

print(f"Frame size      : {_W} x {_H}")
print(f"CSV X range     : {min(_vx):.0f} .. {max(_vx):.0f}   ({100*(max(_vx)-min(_vx))/_W:.0f}% of width)")
print(f"CSV Y range     : {min(_vy):.0f} .. {max(_vy):.0f}   ({100*(max(_vy)-min(_vy))/_H:.0f}% of height)")

if max(_vx) > _W or max(_vy) > _H:
    print("\nWARNING: labels exceed frame bounds.")
elif (max(_vx)-min(_vx))/_W < 0.40 and (max(_vy)-min(_vy))/_H < 0.40:
    print(f"\nWARNING: labels cover less than 40% of frame in both axes.")
    print(f"  If annotated at a smaller size, set SCALE = {_W}/{int(round(max(_vx)))} = {_W/max(_vx):.2f}")
else:
    print("\nOK: coordinates span the frame as expected.")

# ---- no scaling needed ---------------------------------------------------
SCALE = 1.0
print(f"SCALE = {SCALE} — CSV labels used as-is.")

# ---- quick visual spot-check (confirm disc sits on the ball) -------------
import cv2 as _cv2
_vis_idx = [i for i, v in LABELS.items() if v[2]]
_pick    = _vis_idx[::max(1, len(_vis_idx) // 4)][:4]
fig, axes = plt.subplots(1, len(_pick), figsize=(4.5 * len(_pick), 3.4))
for ax, i in zip(np.atleast_1d(axes), _pick):
    im = _cv2.cvtColor(_cv2.imread(FRAME_PATHS[i]), _cv2.COLOR_BGR2RGB)
    x, y, _ = LABELS[i]
    ax.imshow(im)
    ax.add_patch(plt.Circle((x, y), max(_W, _H) * 0.018,
                             fill=False, color="red", lw=2))
    ax.plot(x, y, "r+", ms=12)
    ax.set_title(f"frame {i}  ({x:.0f}, {y:.0f})", fontsize=9)
    ax.axis("off")
plt.suptitle("Red ring MUST sit on the ball.\n"
             "If it is in the wrong place, set SCALE = frame_width / annotation_width above.",
             fontsize=10)
plt.tight_layout(); plt.show()


---
## 7. Dataset and augmentation

`WASBDataset` reproduces `ImageDataset.__getitem__`:

1. affine-warp all 3 frames to 512×288 with **one transform shared across the triplet**;
2. colour-jitter with **one set of factors shared across the triplet** — jittering frames
   independently would inject fake inter-frame change into a model whose whole job is reading
   motion;
3. ImageNet normalisation, then concatenate to 9 channels;
4. build `frames_out` target discs at the affine-mapped, integer-rounded centres;
5. apply paired horizontal flip and random crop to image and target together.

Invisible frames get an all-zero target via the `(-1, -1)` sentinel, which is how the model
learns to output nothing when the ball is occluded.

`build_sequences_flat` slides a 3-frame window across the clip, restricted to the train or test
span, and skips any window containing an unlabelled frame.

In [ ]:
def build_sequences_flat(frame_paths, labels, ranges, frames_in=FRAMES_IN,
                         frames_out=FRAMES_OUT, clip_name="clip"):
    # sliding windows inside [start, end) spans; skips windows with unlabelled frames
    seqs = []
    for start, end in ranges:
        for i in range(start, end - frames_in + 1):
            win = list(range(i, i + frames_in))
            if any(j not in labels for j in win):
                continue
            annos = []
            for j in win[frames_in - frames_out:]:
                x, y, v = labels[j]
                annos.append({"x": float(x), "y": float(y), "visi": bool(v)})
            seqs.append({"frames": [frame_paths[j] for j in win], "annos": annos,
                         "match": clip_name, "clip": clip_name})
    return seqs


def temporal_split(n, train_frac=0.75, gap=None):
    # Contiguous split with a gap so no window straddles the boundary.
    # NOT a substitute for holding out a separate recording - see section 6.
    gap = gap if gap is not None else FRAMES_IN * 4
    cut = int(n * train_frac)
    return [(0, max(0, cut - gap))], [(cut, n)]


TRAIN_FRAC = 0.75
train_ranges, test_ranges = temporal_split(N_FRAMES, TRAIN_FRAC)
print(f"{N_FRAMES} frames -> train {train_ranges}   test {test_ranges}")


def read_image(path):
    return Image.open(path).convert("RGB")

class WASBDataset(Dataset):
    '''Reproduces dataloaders/dataset_loader.py ImageDataset for out_scales=[0].'''
    def __init__(self, cfg, sequences, is_train=True):
        self.seqs = sequences
        self.is_train = is_train
        self.inp_wh = (cfg["model"]["inp_width"], cfg["model"]["inp_height"])
        self.out_wh = (cfg["model"]["out_width"], cfg["model"]["out_height"])
        self.hm_gen = BinaryFixedSizeMapGenerator(cfg["dataloader"]["heatmap"])
        tc = cfg["transform"]["train" if is_train else "test"]
        self.cj = tc["color_jitter"]
        self.flip_p = tc["horizontal_flip"]["p"]
        self.crop_p = tc["crop"]["p"]
        self.max_rescale = tc["crop"]["max_rescale"]
        self.norm = T.Compose([T.ToTensor(),
                               T.Normalize(mean=[0.485, 0.456, 0.406],
                                           std=[0.229, 0.224, 0.225])])

    def __len__(self): return len(self.seqs)

    def _seq_hflip(self, imgs, hms):
        if random.uniform(0, 1) >= self.flip_p: return imgs, hms
        return torch.flip(imgs, [2]), {s: torch.flip(v, [2]) for s, v in hms.items()}

    def _seq_crop(self, imgs, hms):
        '''VERBATIM logic from seq_transforms.RandomCropping: upscale then crop back.'''
        if random.uniform(0, 1) >= self.crop_p: return imgs, hms
        rescale = 1. + self.max_rescale
        _, ih, iw = imgs.shape
        nih, niw = int(ih * rescale), int(iw * rescale)
        r_imgs = TF.resize(imgs, [nih, niw], antialias=True)
        _, hh, hw = hms[SCALE0].shape
        nhh, nhw = int(hh * rescale), int(hw * rescale)
        r_hm = TF.resize(hms[SCALE0], [nhh, nhw], antialias=True)
        x1 = int(round(random.uniform(0, nhw - hw)))
        y1 = int(round(random.uniform(0, nhh - hh)))
        rc_hm = TF.crop(r_hm, top=y1, left=x1, height=hh, width=hw)
        ratio = nih / nhh
        rc_imgs = TF.crop(r_imgs, top=int(y1 * ratio), left=int(x1 * ratio),
                          height=ih, width=iw)
        return rc_imgs, {SCALE0: rc_hm}

    def __getitem__(self, idx):
        seq = self.seqs[idx]
        do_cj = random.uniform(0, 1) < self.cj["p"]
        if do_cj:
            bf = np.random.uniform(max(0, 1 - self.cj["brightness"]), 1 + self.cj["brightness"])
            cf = np.random.uniform(max(0, 1 - self.cj["contrast"]),   1 + self.cj["contrast"])
            sf = np.random.uniform(max(0, 1 - self.cj["saturation"]), 1 + self.cj["saturation"])
            hf = np.random.uniform(-self.cj["hue"], self.cj["hue"])

        trans_in = trans_out = trans_out_inv = None
        imgs = []
        for p in seq["frames"]:
            img = read_image(p)
            if trans_in is None:
                arr = np.asarray(img)
                trans_in      = get_transform(arr, self.inp_wh)
                trans_out     = get_transform(arr, self.out_wh)
                trans_out_inv = get_transform(arr, self.out_wh, inv=1)
            img = Image.fromarray(cv2.warpAffine(np.array(img), trans_in, self.inp_wh,
                                                 flags=cv2.INTER_LINEAR))
            if do_cj:                       # SAME factors for every frame in the triplet
                img = TF.adjust_brightness(img, bf); img = TF.adjust_contrast(img, cf)
                img = TF.adjust_saturation(img, sf); img = TF.adjust_hue(img, hf)
            imgs.append(self.norm(img))
        imgs_t = torch.cat(imgs, dim=0)                       # (9, H, W)

        hm_list, xys, visis = [], [], []
        for a in seq["annos"]:
            if a["visi"]:
                ct = affine_transform(np.array([a["x"], a["y"]], np.float32), trans_out)
                hm = self.hm_gen(self.out_wh, ct.astype(np.int32), binary=True)
            else:
                hm = self.hm_gen(self.out_wh, (-1., -1.))
            hm_list.append(torch.from_numpy(hm)[None])
            xys.append([a["x"], a["y"]]); visis.append(a["visi"])
        hms = {SCALE0: torch.cat(hm_list, dim=0)}             # (frames_out, H, W)

        if self.is_train:
            imgs_t, hms = self._seq_hflip(imgs_t, hms)
            imgs_t, hms = self._seq_crop(imgs_t, hms)
            return imgs_t, hms
        return (imgs_t, hms, torch.from_numpy(trans_out_inv).float(),
                torch.tensor(xys, dtype=torch.float32), torch.tensor(visis))


train_seqs = build_sequences_flat(FRAME_PATHS, LABELS, train_ranges)
test_seqs  = build_sequences_flat(FRAME_PATHS, LABELS, test_ranges)
print(f"train sequences: {len(train_seqs)}\ntest  sequences: {len(test_seqs)}")
assert train_seqs and test_seqs, (
    "empty split - too few frames, or labels missing. For very short clips lower "
    "TRAIN_FRAC or the gap in temporal_split().")

# persistent_workers stops Kaggle's harmless but very noisy
# "AssertionError: can only test a child process" spam, which comes from workers
# being torn down and respawned every epoch.
BATCH_SIZE, NUM_WORKERS = 8, 2
_dl = dict(num_workers=NUM_WORKERS, pin_memory=True,
           persistent_workers=NUM_WORKERS > 0)

train_ds = WASBDataset(CFG, train_seqs, is_train=True)
test_ds  = WASBDataset(CFG, test_seqs,  is_train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, **_dl)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          drop_last=False, **_dl)

xb, yb = next(iter(train_loader))
print(f"\nbatch images  {tuple(xb.shape)}  (B, 3*frames_in, H, W)")
print(f"batch targets {tuple(yb[SCALE0].shape)}  (B, frames_out, H, W)")
print(f"positive pixels per map: {yb[SCALE0].sum().item()/ (xb.shape[0]*FRAMES_OUT):.1f}")

---
## 8. Sanity checks

Two things to confirm before spending GPU time.

**Targets land on the ball.** Below, the warped frame with its target disc overlaid. If the
disc is off the ball, the CSVs are misaligned and nothing downstream can fix it.

**The disc radius suits your ball.** `sigmas: [2.5]` was tuned for the tennis/table-tennis
datasets. Measure the ball's apparent radius in the *warped* 512×288 frame and set the radius
to roughly match. This is the one hyperparameter here with a physical meaning.

In [ ]:
def denorm(t):
    m = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    s = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    return (t * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

imgs_t, hms = train_ds[len(train_ds) // 2]
fig, ax = plt.subplots(1, 3, figsize=(17, 3.4))
ax[0].imshow(denorm(imgs_t[6:9])); ax[0].set_title("warped frame 3 of 3"); ax[0].axis("off")
ax[1].imshow(hms[SCALE0][-1], cmap="inferno"); ax[1].set_title("target heatmap"); ax[1].axis("off")
ax[2].imshow(denorm(imgs_t[6:9]))
ax[2].imshow(hms[SCALE0][-1], cmap="inferno", alpha=0.55)
ax[2].set_title("overlay - disc MUST sit on the ball"); ax[2].axis("off")
plt.tight_layout(); plt.show()

pos = hms[SCALE0][-1].sum().item()
print(f"positive pixels: {pos:.0f}  (r={CFG['dataloader']['heatmap']['sigmas'][0]} "
      f"-> pi*r^2 ~ {math.pi*CFG['dataloader']['heatmap']['sigmas'][0]**2:.0f})")
print(textwrap.dedent('''
    To retune for your footage: measure the ball's radius in the WARPED 512x288
    frame (zoom into the overlay above), then set

        CFG["dataloader"]["heatmap"]["sigmas"] = [that radius]

    and rebuild the datasets. Default 2.5 suits a ball ~5 px across.
'''))

In [ ]:
# Zero-shot check: do the tennis weights already fire on pickleball?
model.eval()
with torch.no_grad():
    probe_imgs, probe_hms, _, _, _ = test_ds[len(test_ds) // 2]
    logits = model(probe_imgs[None].to(DEVICE))[SCALE0]
    prob   = torch.sigmoid(logits)[0, -1].cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(17, 3.4))
ax[0].imshow(denorm(probe_imgs[6:9])); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(probe_hms[SCALE0][-1], cmap="inferno"); ax[1].set_title("target"); ax[1].axis("off")
im = ax[2].imshow(prob, cmap="inferno", vmin=0, vmax=1)
ax[2].set_title(f"tennis weights, zero-shot (max {prob.max():.3f})"); ax[2].axis("off")
plt.colorbar(im, ax=ax[2], fraction=0.03); plt.tight_layout(); plt.show()

print(f"peak response {prob.max():.4f} vs detection threshold "
      f"{CFG['detector']['postprocessor']['score_threshold']}")
print(textwrap.dedent("""
    Read this together with the TARGET panel in the middle, not on its own.
    A low peak only means fine-tuning has work to do IF the bright spot in the
    target panel is actually on the ball. If the target is sitting on empty
    court, stop here - go back to section 6b. No amount of training fixes a
    label that points at the wrong place.
"""))

---
## 8b. Evaluator (moved up)

`TracknetV2Postprocessor` and `Evaluator` normally live in §10, but §9 measures detection quality every epoch so they must be defined first.
Nothing is changed — just the position in the notebook.

In [ ]:
class TracknetV2Postprocessor:
    '''VERBATIM logic from detectors/postprocessor.py (concomp branch).'''
    def __init__(self, cfg):
        pp = cfg["detector"]["postprocessor"]
        self._score_threshold = pp["score_threshold"]
        self._use_hm_weight   = pp["use_hm_weight"]

    def _detect_blob_concomp(self, hm):
        xys, scores = [], []
        if np.max(hm) > self._score_threshold:
            _, hm_th = cv2.threshold(hm, self._score_threshold, 1, cv2.THRESH_BINARY)
            n_labels, labels = cv2.connectedComponents(hm_th.astype(np.uint8))
            for m in range(1, n_labels):
                ys, xs = np.where(labels == m)
                ws = hm[ys, xs]
                if self._use_hm_weight:
                    score = ws.sum()
                    x = np.sum(xs * ws) / np.sum(ws)
                    y = np.sum(ys * ws) / np.sum(ws)
                else:
                    score = ws.shape[0]
                    x, y = xs.mean(), ys.mean()
                xys.append(np.array([x, y])); scores.append(float(score))
        return xys, scores

    def run(self, logits, affine_inv):
        '''logits (B, S, H, W) -> per (b, s) list of dets in ORIGINAL pixel coords.'''
        hms = torch.sigmoid(logits).cpu().numpy()
        inv = affine_inv.cpu().numpy()
        out = defaultdict(dict)
        for b in range(hms.shape[0]):
            for s in range(hms.shape[1]):
                xys, scores = self._detect_blob_concomp(hms[b, s])
                out[b][s] = {"xys": [affine_transform(xy.astype(np.float32), inv[b])
                                     for xy in xys],
                             "scores": scores}
        return out


class Evaluator:
    '''VERBATIM logic from utils/evaluator.py.'''
    def __init__(self, cfg):
        self._dist_threshold = cfg["runner"]["eval"]["dist_threshold"]
        self._tp = self._fp1 = self._fp2 = self._tn = self._fn = 0
        self._ses = []

    def eval_single_frame(self, xy_pred, visi_pred, xy_gt, visi_gt):
        if visi_gt:
            if visi_pred:
                d = np.linalg.norm(np.array(xy_pred) - np.array(xy_gt))
                if d < self._dist_threshold: self._tp += 1
                else:                        self._fp1 += 1
                self._ses.append(d ** 2)
            else:
                self._fn += 1
        else:
            if visi_pred: self._fp2 += 1
            else:         self._tn += 1

    @property
    def fp_all(self): return self._fp1 + self._fp2
    @property
    def prec(self):
        d = self._tp + self.fp_all;  return self._tp / d if d else 0.
    @property
    def recall(self):
        d = self._tp + self._fn;     return self._tp / d if d else 0.
    @property
    def f1(self):
        p, r = self.prec, self.recall
        return 2 * p * r / (p + r) if (p + r) else 0.
    @property
    def accuracy(self):
        d = self._tp + self._tn + self.fp_all + self._fn
        return (self._tp + self._tn) / d if d else 0.
    @property
    def rmse(self):
        return math.sqrt(np.mean(self._ses)) if self._ses else float("nan")

    def summary(self, name=""):
        print(f"--- {name} (dist_threshold={self._dist_threshold} px) ---")
        print(f"  TP {self._tp:5d} | FP1 {self._fp1:5d} | FP2 {self._fp2:5d} | "
              f"FN {self._fn:5d} | TN {self._tn:5d}")
        print(f"  precision {self.prec:.4f}  recall {self.recall:.4f}  F1 {self.f1:.4f}")
        print(f"  accuracy  {self.accuracy:.4f}  RMSE {self.rmse:.2f} px")
        return {"tp": self._tp, "fp1": self._fp1, "fp2": self._fp2, "fn": self._fn,
                "tn": self._tn, "prec": self.prec, "recall": self.recall,
                "f1": self.f1, "accuracy": self.accuracy, "rmse": self.rmse}


@torch.no_grad()
def evaluate(model, loader, cfg, name=""):
    model.eval()
    post, ev = TracknetV2Postprocessor(cfg), Evaluator(cfg)
    for imgs, hms, inv, xys_gt, visis_gt in tqdm(loader, desc=f"[EVAL {name}]", leave=False):
        logits = model(imgs.to(DEVICE))[SCALE0]
        res = post.run(logits, inv)
        for b in range(imgs.shape[0]):
            for s in range(logits.shape[1]):
                dets = res[b][s]
                if dets["scores"]:
                    k = int(np.argmax(dets["scores"]))
                    xy_pred, visi_pred = dets["xys"][k], True
                else:
                    xy_pred, visi_pred = (0., 0.), False
                ev.eval_single_frame(xy_pred, visi_pred,
                                     xys_gt[b, s].numpy(), bool(visis_gt[b, s]))
    return ev.summary(name)

# ---------------------------------------------------------------------------
# helpers used by per-epoch tracking inside fit()
# ---------------------------------------------------------------------------
@torch.no_grad()
def collect_predictions(model, loader, cfg):
    model.eval()
    post = TracknetV2Postprocessor(cfg)
    records = []
    for imgs, hms, inv, xys_gt, visis_gt in loader:
        logits = model(imgs.to(DEVICE))[SCALE0]
        res = post.run(logits, inv)
        for b in range(imgs.shape[0]):
            for s in range(logits.shape[1]):
                d = res[b][s]
                if d["scores"]:
                    k = int(np.argmax(d["scores"]))
                    xy_pred, visi_pred = d["xys"][k], True
                else:
                    xy_pred, visi_pred = (0.0, 0.0), False
                records.append((xy_pred, visi_pred,
                                xys_gt[b, s].numpy(), bool(visis_gt[b, s])))
    return records


def score_records(records, dist_threshold):
    ev = Evaluator({"runner": {"eval": {"dist_threshold": dist_threshold}}})
    for xy_pred, visi_pred, xy_gt, visi_gt in records:
        ev.eval_single_frame(xy_pred, visi_pred, xy_gt, visi_gt)
    return {"tp": ev._tp, "fp1": ev._fp1, "fp2": ev._fp2,
            "fn": ev._fn, "tn": ev._tn,
            "prec": ev.prec, "recall": ev.recall, "f1": ev.f1,
            "accuracy": ev.accuracy, "rmse": ev.rmse}


THRESHOLDS  = [2, 4, 6, 8, 12]
MAIN_THRESH = 4
print(f"thresholds {THRESHOLDS} px  |  checkpoint selected on F1@{MAIN_THRESH}px")


---
## 9. Train

`train_epoch` and `test_epoch` follow `runners/runner_utils.py`. Note the model emits **logits**
and the sigmoid lives inside the loss, so never apply sigmoid before `criterion`.

**Staged fine-tuning.** The paper's 3e-4 is for training from scratch. On a few thousand
pickleball frames that is high enough to wash out the pretrained features in the first epoch, so
stage 1 freezes the stem and stage 1 (`conv1 bn1 conv2 bn2 layer1` — the low-level edge and
colour filters, which transfer from tennis unchanged) and runs at 1e-4; stage 2 unfreezes
everything at 3e-5. Raise the epoch counts for a real dataset — the paper uses 30.

Best checkpoint is selected on test loss.

In [ ]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val = val; self.sum += val * n; self.count += n
        self.avg = self.sum / max(self.count, 1)


def train_epoch(epoch, model, loader, criterion, optimizer, device):
    meter = AverageMeter(); model.train(); t0 = time.time()
    for imgs, hms in tqdm(loader, desc=f"[TRAIN {epoch}]", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        hms  = {s: v.to(device, non_blocking=True) for s, v in hms.items()}
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, hms)
        loss.backward(); optimizer.step()
        meter.update(loss.item(), preds[SCALE0].size(0))
    print(f"(TRAIN) Epoch {epoch}  loss {meter.avg:.6f}  {time.time()-t0:.0f}s")
    return meter.avg


@torch.no_grad()
def test_epoch(epoch, model, loader, criterion, device):
    meter = AverageMeter(); model.eval(); t0 = time.time()
    for imgs, hms, *_ in tqdm(loader, desc=f"[TEST {epoch}]", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        hms  = {s: v.to(device, non_blocking=True) for s, v in hms.items()}
        preds = model(imgs)
        meter.update(criterion(preds, hms).item(), preds[SCALE0].size(0))
    print(f"(TEST)  Epoch {epoch}  loss {meter.avg:.6f}  {time.time()-t0:.0f}s")
    return meter.avg


def set_frozen(model, prefixes):
    froz = tot = 0
    for n, p in model.named_parameters():
        tot += p.numel()
        p.requires_grad = not (prefixes and n.startswith(prefixes))
        if not p.requires_grad: froz += p.numel()
    print(f"frozen {froz:,}/{tot:,} params ({100*froz/tot:.1f}%)")


EVAL_EVERY = 1   # measure detection every N epochs; raise to 2-3 if slow


def fit(model, epochs, lr, freeze=(), tag="run"):
    """Train and record detection metrics every epoch.

    Two improvements over a plain loop:
    - checkpoint is saved on best F1, not best loss (they disagree)
    - cosine LR decay so weights settle precisely near the end
    """
    set_frozen(model, tuple(freeze))
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt   = torch.optim.Adam(trainable, lr=lr,
                             weight_decay=CFG["optimizer"]["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=epochs, eta_min=lr * 0.05)

    out = f"{RUNS}/{tag}"; os.makedirs(out, exist_ok=True)
    hist, best_f1 = [], -1.0

    for ep in range(1, epochs + 1):
        tr = train_epoch(ep, model, train_loader, criterion, opt, DEVICE)
        te = test_epoch(ep, model, test_loader,  criterion, DEVICE)
        sched.step()

        row = {"epoch": ep, "train_loss": tr, "test_loss": te,
               "lr": opt.param_groups[0]["lr"]}

        if ep % EVAL_EVERY == 0 or ep == epochs:
            t0 = time.time()
            recs = collect_predictions(model, test_loader, CFG)
            for thr in THRESHOLDS:
                for k, v in score_records(recs, thr).items():
                    row[f"{k}@{thr}"] = v
            m = {k.split("@")[0]: v for k, v in row.items()
                 if k.endswith(f"@{MAIN_THRESH}")}
            print(f"        eval {time.time()-t0:.0f}s | "
                  f"P {m['prec']:.3f}  R {m['recall']:.3f}  "
                  f"F1 {m['f1']:.3f}  acc {m['accuracy']:.3f}  "
                  f"RMSE {m['rmse']:.1f}px | "
                  f"TP {m['tp']}  FP1 {m['fp1']}  FP2 {m['fp2']}  FN {m['fn']}")
            if m["f1"] > best_f1:
                best_f1 = m["f1"]
                torch.save({"model_state_dict": model.state_dict(),
                            "epoch": ep, "f1": best_f1},
                           f"{out}/best_model.pth.tar")
                print(f"        -> best F1 {best_f1:.4f} saved")
        hist.append(row)

    print(f"\nbest F1@{MAIN_THRESH}px = {best_f1:.4f}")
    return f"{out}/best_model.pth.tar", hist


In [ ]:
# T4 timing: ~3.7 min training + ~75 s eval per epoch
# 3 + 25 epochs  ≈  2 hours total
STAGE1_EPOCHS = 3
STAGE2_EPOCHS = 25

STEM = ("conv1.", "bn1.", "conv2.", "bn2.", "layer1.")

print("=" * 70)
print("STAGE 1 — frozen stem, lr 1e-4")
print("=" * 70)
ckpt1, hist1 = fit(model, STAGE1_EPOCHS, 1e-4, freeze=STEM, tag="stage1")

print("\n" + "=" * 70)
print("STAGE 2 — full unfreeze, lr 3e-5 with cosine decay")
print("=" * 70)
model.load_state_dict(load_state_dict_clean(ckpt1))
ckpt2, hist2 = fit(model, STAGE2_EPOCHS, 3e-5, freeze=(), tag="stage2")
FINAL_CKPT = ckpt2

# combine both stages into one dataframe for the charts
hist_df = pd.DataFrame(
    [dict(r, stage=1) for r in hist1] +
    [dict(r, stage=2, epoch=r["epoch"] + STAGE1_EPOCHS) for r in hist2])
hist_df.to_csv(f"{RUNS}/history.csv", index=False)
print(f"\nhistory saved -> {RUNS}/history.csv")


In [ ]:
# ---- build charts only if we have per-epoch metric columns ----------------
ev_df  = hist_df.dropna(subset=[f"f1@{MAIN_THRESH}"])
split  = STAGE1_EPOCHS + 0.5     # dotted line = stage 1 -> stage 2 handover

# ---- four-panel chart -------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# panel 1 — loss
ax = axes[0, 0]
ax.plot(hist_df.epoch, hist_df.train_loss, "o-", ms=3, label="train")
ax.plot(hist_df.epoch, hist_df.test_loss,  "s-", ms=3, label="test")
ax.set_title("Quality focal loss"); ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.legend(); ax.grid(alpha=.3)

# panel 2 — four detection metrics on one axis
ax = axes[0, 1]
ax.plot(ev_df.epoch, ev_df[f"f1@{MAIN_THRESH}"],       "o-", ms=4, lw=2, label="F1")
ax.plot(ev_df.epoch, ev_df[f"recall@{MAIN_THRESH}"],   "s-", ms=3,       label="recall")
ax.plot(ev_df.epoch, ev_df[f"prec@{MAIN_THRESH}"],     "^-", ms=3,       label="precision")
ax.plot(ev_df.epoch, ev_df[f"accuracy@{MAIN_THRESH}"], "d-", ms=3,       label="accuracy")
ax.set_title(f"Detection metrics  @{MAIN_THRESH} px")
ax.set_xlabel("epoch"); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=.3)

# panel 3 — F1 at every threshold (shows whether ruler or model is the limit)
ax = axes[1, 0]
for thr in THRESHOLDS:
    ax.plot(ev_df.epoch, ev_df[f"f1@{thr}"], "o-", ms=3, label=f"{thr} px")
ax.set_title("F1 by distance threshold\n"
             "(if lines are far apart, precision is limited by grid resolution)")
ax.set_xlabel("epoch"); ax.set_ylabel("F1"); ax.set_ylim(0, 1)
ax.legend(title="tolerance", fontsize=8); ax.grid(alpha=.3)

# panel 4 — RMSE
ax = axes[1, 1]
ax.plot(ev_df.epoch, ev_df[f"rmse@{MAIN_THRESH}"], "o-", ms=3, color="tab:red")
ax.axhline(MAIN_THRESH, ls="--", lw=1, color="gray")
ax.text(ev_df.epoch.iloc[0], MAIN_THRESH * 1.12,
        f"{MAIN_THRESH} px pass mark", fontsize=8, color="gray")
ax.set_title("Localisation error (RMSE)")
ax.set_xlabel("epoch"); ax.set_ylabel("px  (original frame)"); ax.grid(alpha=.3)

for a in axes.ravel():
    a.axvline(split, ls=":", lw=1, color="gray")
plt.suptitle("Dotted line = stage 1 → stage 2 handover", fontsize=10)
plt.tight_layout(); plt.show()

# ---- stacked area: what kind of error remains each epoch ------------------
fig, ax = plt.subplots(figsize=(10, 4))
ax.stackplot(ev_df.epoch,
             ev_df[f"tp@{MAIN_THRESH}"],  ev_df[f"fp1@{MAIN_THRESH}"],
             ev_df[f"fp2@{MAIN_THRESH}"], ev_df[f"fn@{MAIN_THRESH}"],
             labels=["TP — correct",
                     "FP1 — found but mislocated (localisation error)",
                     "FP2 — hallucinated (ball not there)",
                     "FN — missed (ball not found)"],
             colors=["#639922", "#EF9F27", "#E24B4A", "#888780"])
ax.axvline(split, ls=":", lw=1, color="white")
ax.set_title(f"Error composition per epoch  @{MAIN_THRESH} px\n"
             "A shrinking FP1 bar = localisation improving")
ax.set_xlabel("epoch"); ax.set_ylabel("frames")
ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

# ---- numeric table ---------------------------------------------------------
cols = ["epoch", "stage", "train_loss", "test_loss",
        f"prec@{MAIN_THRESH}", f"recall@{MAIN_THRESH}",
        f"f1@{MAIN_THRESH}", f"accuracy@{MAIN_THRESH}", f"rmse@{MAIN_THRESH}"]
print(hist_df[cols].to_string(index=False, float_format=lambda v: f"{v:.4f}"))


---
## 10. Evaluate

Loss is not the metric you care about. `TracknetV2Postprocessor` decodes heatmaps into
coordinates, and `Evaluator` scores them the way the paper reports.

**Decoding** (`detectors/postprocessor.py`): threshold at 0.5, take connected components, and
for each one compute the **heatmap-weighted centroid** — a sub-pixel estimate, strictly better
than the arg-max pixel. Coordinates are then mapped back through the inverse affine to original
pixels.

One trap: with `use_hm_weight: True` the returned blob **score is the summed heatmap mass of
the component, not a probability**. It is unbounded and routinely runs into the thousands, so
it ranks candidates within a frame but must never be thresholded as if it lived in [0,1]. The
0.5 threshold is applied to the *pixel* probabilities, before components are formed.

**Metric** (`utils/evaluator.py`) splits errors in a way plain precision/recall hides:

| | Meaning |
|---|---|
| TP | ball visible, detected within `dist_threshold` (4 px) |
| FP1 | ball visible, detected, but **too far** — a localisation failure |
| FP2 | ball not visible, yet something was detected — a hallucination |
| FN | ball visible, nothing detected — a miss |
| TN | ball not visible, nothing detected |

FP1 and FP2 are different bugs with different fixes. A high FP1 means the detector finds the
ball but localises poorly — revisit the disc radius. A high FP2 means it fires on court lines
or shoes — more negatives, more augmentation.

Also run the **zero-shot tennis baseline**. If fine-tuning does not clearly beat it, something
upstream is wrong and no amount of extra epochs will help.

In [ ]:
zero = build_model(CFG).to(DEVICE)
zero.load_state_dict(load_state_dict_clean(INIT_CKPT))
m_zero = evaluate(zero, test_loader, CFG, "zero-shot tennis")

print()
model.load_state_dict(load_state_dict_clean(FINAL_CKPT))
m_ft = evaluate(model, test_loader, CFG, "fine-tuned pickleball")

print("\n" + "=" * 58)
print(f"{'metric':<12}{'zero-shot':>13}{'fine-tuned':>13}{'delta':>13}")
print("-" * 58)
for k in ("prec", "recall", "f1", "accuracy", "rmse"):
    d = m_ft[k] - m_zero[k]
    print(f"{k:<12}{m_zero[k]:>13.4f}{m_ft[k]:>13.4f}{d:>+13.4f}")
print("=" * 58)
print(textwrap.dedent('''
    If fine-tuned does not clearly win, check in this order:
      1. target overlay in section 8 - misaligned CSVs beat every other cause
      2. disc radius vs your ball size
      3. split leakage - whole rallies must not straddle train and test
      4. too few epochs (the paper uses 30)
    High FP1 = poor localisation (radius). High FP2 = firing on lines/shoes
    (needs more data and augmentation).
'''))

---
## 10b. Per-frame merging and threshold sweep

Each frame appears in three overlapping 3-frame windows, so §10 scores every
frame three times. This section collapses them — one prediction per frame — and
shows a threshold sweep so you can see where your model really sits.


In [ ]:
@torch.no_grad()
def collect_per_frame(model, loader, seqs, cfg):
    """Group the 3 overlapping predictions per frame."""
    model.eval()
    post = TracknetV2Postprocessor(cfg)
    frames, idx = {}, 0
    for imgs, hms, inv, xys_gt, visis_gt in tqdm(loader, desc="[merge]", leave=False):
        logits = model(imgs.to(DEVICE))[SCALE0]
        res = post.run(logits, inv)
        for b in range(imgs.shape[0]):
            seq = seqs[idx]; idx += 1
            n_out = logits.shape[1]
            paths = seq["frames"][len(seq["frames"]) - n_out:]
            for s in range(n_out):
                rec = frames.setdefault(
                    paths[s], {"cands": [],
                               "gt":   tuple(xys_gt[b, s].numpy()),
                               "visi": bool(visis_gt[b, s])})
                d = res[b][s]
                if d["scores"]:
                    k = int(np.argmax(d["scores"]))
                    rec["cands"].append((np.asarray(d["xys"][k], float),
                                        float(d["scores"][k])))
    return frames


def merge(frames, how="median"):
    out = []
    for rec in frames.values():
        cands = rec["cands"]
        if not cands:
            xy, visi = (0.0, 0.0), False
        else:
            visi = True
            if   how == "best":   xy = max(cands, key=lambda c: c[1])[0]
            elif how == "mean":   xy = np.mean([c[0] for c in cands], axis=0)
            else:                 xy = np.median(np.stack([c[0] for c in cands]), axis=0)
        out.append((xy, visi, np.asarray(rec["gt"], float), rec["visi"]))
    return out


model.load_state_dict(load_state_dict_clean(FINAL_CKPT))
frames = collect_per_frame(model, test_loader, test_seqs, CFG)
n_c = [len(v["cands"]) for v in frames.values()]
print(f"{len(frames)} distinct frames  "
      f"(section 10 scored {len(test_seqs)*FRAMES_OUT} rows)\n")

# ---- compare merge strategies --------------------------------------------
_thr = [2, 4, 6, 8]
print(f"{'strategy':>12} " + " ".join(f"{'F1@'+str(t):>9}" for t in _thr)
      + f" {'RMSE':>9} {'recall':>8}")
print("-" * 66)
results = {}
for how in ("per-window", "best", "mean", "median"):
    if how == "per-window":
        recs = []
        for rec in frames.values():
            for xy, sc in rec["cands"]:
                recs.append((xy, True, np.asarray(rec["gt"], float), rec["visi"]))
            if not rec["cands"]:
                recs.append(((0.,0.), False, np.asarray(rec["gt"], float), rec["visi"]))
    else:
        recs = merge(frames, how)
    row = [score_records(recs, t) for t in _thr]
    results[how] = row
    print(f"{how:>12} " + " ".join(f"{m['f1']:9.4f}" for m in row)
          + f" {row[1]['rmse']:9.2f} {row[1]['recall']:8.4f}")

BEST_MERGE = max((h for h in results if h != "per-window"),
                 key=lambda h: results[h][1]["f1"])
print(f"\nbest merge strategy: {BEST_MERGE}")

# ---- threshold sweep on best merge ---------------------------------------
recs  = merge(frames, BEST_MERGE)
sweep = [(t, score_records(recs, t))
         for t in [1, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 30]]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot([t for t,_ in sweep], [m["f1"] for _,m in sweep], "o-")
ax.axvline(4, ls="--", color="gray", lw=1)
ax.text(4.3, 0.08, "paper's 4 px", fontsize=8, color="gray")
ax.set_xlabel("distance threshold (px, original frame)")
ax.set_ylabel("F1"); ax.set_ylim(0, 1)
ax.set_title(f"F1 vs tolerance  ({BEST_MERGE} merge)\n"
             "A steep rise = model is right, tolerance is the constraint")
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

_s = 1920 / CFG["model"]["inp_width"]
print(f"\n{'thresh':>7} {'= hm px':>8} {'F1':>8} {'prec':>8} "
      f"{'recall':>8} {'TP':>6} {'FP1':>6} {'FN':>5}")
print("-" * 64)
for t, m in sweep:
    print(f"{t:7d} {t/_s:8.2f} {m['f1']:8.4f} {m['prec']:8.4f} "
          f"{m['recall']:8.4f} {m['tp']:6d} {m['fp1']:6d} {m['fn']:5d}")


---
## 11. Inference and export

`predict_triplet` runs the whole path on any 3 consecutive frames: affine warp → model →
threshold → connected components → weighted centroid → inverse affine back to original pixels.

For ONNX, `forward` returns a dict, which the exporter rejects — the wrapper unwraps scale 0 and
folds the sigmoid in so downstream code receives probabilities. From there, TensorRT FP16 on a
Jetson or OpenVINO on a Pi. After INT8 quantisation, check **localisation error (RMSE)**, not
just F1: heatmap regression degrades in a way detection-rate metrics hide.

In [ ]:
IMNORM = T.Compose([T.ToTensor(),
                    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

@torch.no_grad()
def predict_triplet(model, frame_paths, cfg=CFG, score_threshold=None):
    '''3 consecutive frame paths -> [(x, y, score) or (None, None, 0.0)] in ORIGINAL pixels.'''
    assert len(frame_paths) == FRAMES_IN
    pp = dict(cfg["detector"]["postprocessor"])
    if score_threshold is not None: pp["score_threshold"] = score_threshold
    post = TracknetV2Postprocessor({"detector": {"postprocessor": pp}})

    trans_in = trans_inv = None
    chans = []
    for p in frame_paths:
        img = read_image(p); arr = np.asarray(img)
        if trans_in is None:
            trans_in  = get_transform(arr, (INP_W, INP_H))
            trans_inv = get_transform(arr, (OUT_W, OUT_H), inv=1)
        w = cv2.warpAffine(np.array(img), trans_in, (INP_W, INP_H), flags=cv2.INTER_LINEAR)
        chans.append(IMNORM(Image.fromarray(w)))
    x = torch.cat(chans, 0)[None].to(DEVICE)

    model.eval()
    logits = model(x)[SCALE0]
    res = post.run(logits, torch.from_numpy(trans_inv).float()[None])
    out = []
    for s in range(logits.shape[1]):
        d = res[0][s]
        if d["scores"]:
            k = int(np.argmax(d["scores"]))
            out.append((float(d["xys"][k][0]), float(d["xys"][k][1]), float(d["scores"][k])))
        else:
            out.append((None, None, 0.0))
    return out, torch.sigmoid(logits)[0].cpu().numpy()


seq = test_seqs[len(test_seqs) // 2]
dets, probs = predict_triplet(model, seq["frames"])
for i, (x, y, s) in enumerate(dets):
    gt = seq["annos"][i]
    if x is None:
        print(f"  frame {i}: no detection            gt=({gt['x']:.0f},{gt['y']:.0f})")
    else:
        print(f"  frame {i}: ({x:7.1f},{y:7.1f}) score {s:7.2f}  "
              f"gt=({gt['x']:.0f},{gt['y']:.0f})  err {np.hypot(x-gt['x'], y-gt['y']):5.1f} px")

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
vis = cv2.cvtColor(cv2.imread(seq["frames"][-1]), cv2.COLOR_BGR2RGB)
ax[0].imshow(vis)
gt = seq["annos"][-1]
ax[0].add_patch(plt.Circle((gt["x"], gt["y"]), 22, fill=False, color="lime", lw=2, label="GT"))
if dets[-1][0] is not None:
    ax[0].add_patch(plt.Circle(dets[-1][:2], 14, fill=False, color="red", lw=2, label="pred"))
ax[0].legend(loc="upper right"); ax[0].set_title("green = ground truth, red = prediction")
ax[0].axis("off")
ax[1].imshow(probs[-1], cmap="inferno", vmin=0, vmax=1)
ax[1].set_title(f"predicted heatmap (max {probs[-1].max():.3f})"); ax[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
try:
    import onnxruntime  # noqa: F401
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "onnxruntime", "onnxscript"], check=False)


class ExportWrapper(nn.Module):
    '''forward() returns a dict, which ONNX rejects. Unwrap scale 0 and fold in the sigmoid.'''
    def __init__(self, net, scale=SCALE0):
        super().__init__(); self.net, self.scale = net, scale
    def forward(self, x):
        return torch.sigmoid(self.net(x)[self.scale])

wrapped = ExportWrapper(build_model(CFG)).eval()
wrapped.net.load_state_dict(load_state_dict_clean(FINAL_CKPT))
dummy = torch.randn(1, 3 * FRAMES_IN, INP_H, INP_W)
with torch.no_grad():
    print("output:", tuple(wrapped(dummy).shape))

# Always save the torch checkpoint first - it is the deliverable that matters.
torch.save({"model_state_dict": load_state_dict_clean(FINAL_CKPT), "cfg": CFG},
           f"{WORK}/wasb_pickleball_final.pth.tar")
print(f"saved {WORK}/wasb_pickleball_final.pth.tar")

# torch >= 2.6 defaults to the dynamo exporter, which needs onnxscript. Fall back to
# the legacy tracer when it is unavailable rather than failing the whole notebook.
ONNX_PATH = f"{WORK}/wasb_pickleball.onnx"
common = dict(input_names=["frames"], output_names=["heatmap"],
              dynamic_axes={"frames": {0: "batch"}, "heatmap": {0: "batch"}},
              opset_version=17, do_constant_folding=True)
ok = False
for kwargs in ({"dynamo": False}, {}):
    try:
        torch.onnx.export(wrapped, dummy, ONNX_PATH, **common, **kwargs)
        ok = True
        print(f"exported ({'legacy tracer' if kwargs else 'default exporter'})")
        break
    except (TypeError, ImportError, ModuleNotFoundError) as e:
        print(f"  export attempt failed: {type(e).__name__}: {str(e)[:120]}")
    except Exception as e:
        print(f"  export attempt failed: {type(e).__name__}: {str(e)[:120]}")

if ok:
    print(f"{ONNX_PATH}  ({os.path.getsize(ONNX_PATH)/1e6:.1f} MB)")
    try:
        import onnxruntime as ort
        s = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
        o = s.run(None, {"frames": dummy.numpy()})[0]
        with torch.no_grad():
            r = wrapped(dummy).numpy()
        print(f"onnxruntime OK | max |torch-onnx| = {np.abs(r-o).max():.2e}")
    except ImportError:
        print("onnxruntime not installed; skipped numerical check")
else:
    print("\nONNX export unavailable in this environment. Run:")
    print("    !pip install -q onnxscript onnxruntime")
    print("and re-run this cell. The .pth.tar checkpoint above is already saved.")

---
## 12. Notes

**Do first**
- Confirm the red ring in §6b sits on the ball, and that the disc in §8 sits on the ball. Those
  two overlays are the only thing standing between you and training on wrong labels.
- Run the zero-shot row of §10 BEFORE training. It takes about 40 seconds and tells you whether
  the labels are right with no GPU time spent. Non-zero TP and an RMSE in the tens of pixels
  means go; TP 0 with an RMSE in the hundreds means stop and fix §6b.
- Retune the disc radius in §8 for your ball's apparent size.
- STAGE2_EPOCHS is 25; the paper uses 30.

**Then**
- Bake off tennis vs badminton initialisation — 3 epochs each is enough to see the difference.
- Tune the detection threshold on your own precision/recall trade-off; 0.5 is the paper's
  default, not a law.
- The paper's third contribution, **temporal-consistency inference**, is not implemented here.
  Because the model is MIMO, each frame is predicted in several overlapping windows, and those
  predictions can be combined and linked into tracks — the repo's `OnlineTracker` scores
  candidates by proximity to a constant-acceleration prediction from the previous three frames.
  Worth adding once single-frame detection is solid; it mostly recovers frames where the
  per-frame peak is ambiguous.

**Gotchas already handled here, which will bite if you refactor**
- The model outputs **logits**; the sigmoid lives in the loss. Applying it twice trains garbage.
- `_sigmoid` clamps to `[1e-4, 1-1e-4]`. Remove the clamp and the loss returns NaN.
- The input warp is affine, not `cv2.resize`.
- `sigmas` is a disc **radius**, not a Gaussian sigma.
- Colour jitter uses the **same factors across a triplet**; independent jitter fakes motion.
- Label coordinates must be in ORIGINAL frame pixels. The affine warp downscales them; if the
  CSV was annotated on smaller frames they get downscaled twice and land in the corner. §6b
  measures and corrects this.
- Checkpoints saved under `DataParallel` carry a `module.` prefix — `load_state_dict_clean`
  strips it.